# 1.2 - Data cleaning (`accidentes.csv`)

## Cleaning decisions

**Duplicates**

- 100 IDs appear twice → keep the more complete row (then the earlier one)

**Kept as recorded**

- `numero_incidencia`: incident id. When an id appears twice, one row is kept (the fuller one).
- `fecha`: original date string, including the `0:00` time.
- `hora`: original clock time (`HH:MM:SS`).
- `localizacion`, `eje`: location codes, unchanged.
- `descripcion`: free text, unchanged, blanks included.
- `unidad_samur_id`: unchanged, blanks included.

**Categoricals.** Junk (`(En blanco)`, `0`, `_`, empty) becomes NaN. Other values collapse to one spelling.

- `estado_firme`: normalize(Lowercase, strip accents, punctuation) `seco` and `seco y limpio` (any case) → `Seco y limpio`; `mojado` → `Mojado`; `humedo` → `Húmedo`. `Otros` stays.
- `tipo_via`: case variants → `TRONCO`, `VIA DE SERVICIO`, `RAMAL`, `TRANSFER`, `TABLERO`. `Túnel` / `TUNEL` → `TUNEL`.
- `carretera`: uppercase, hyphens removed, removed. `M-607` and `M607` → `M607`. Blank → NaN.
- `ubicacion`: already two labels, `SUPERFICIE` and `TÚNEL`. Unchanged.
- `tipo_colision`: 80 blanks → NaN. The five collision labels stay.
- `tipo_accidente`: six labels, one spelling each. Unchanged.
- `circulacion`: blank, `0`, and `_` → NaN. `FLUIDA`, `CONGESTIONADA`, `DENSA` stay.
- `distrito`: Unchanged.

**Counts.** Stored as numbers. `-1` means missing and becomes NaN.

- `vehiculos_implicados`: 15 values of `-1` → NaN. The 17 zeros stay.
- `heridos_leves`: 15 values of `-1` → NaN.
- `motos_implicadas`, `pesados_implicados`, `heridos_graves`, `victimas_mortales`: no `-1` values. Stored as numbers.

**New columns**

- `fecha_calendario`: calendar date parsed from `fecha` (the `0:00` time is dropped). Tries day/month/year, then year-month-day, then month/day/year.
- `hora_decimal`: clock time from `hora`, as hours + minutes/60. Seconds are ignored. `15:11:00` → `15.183`.
- `grave`: 1 if `heridos_graves > 0` or `victimas_mortales > 0`, otherwise 0.

In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/accidentes.csv")
OUT_PATH = Path("./data/accidentes_clean.csv")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print("loaded:", DATA_PATH.resolve())
print("shape:", df.shape)
df.head()


loaded: /Users/lukas/dev/UC3M/Project-UC3M/data/accidentes.csv
shape: (2062, 21)


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,motos_implicadas,pesados_implicados,estado_firme,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id
0,01/01/2024 0:00,1:30:48,20240101-0001,08NL10,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,NaN,1,0,0,SEÑALIZAR ACCIDENTE POR ALCANCE EN CARRIL CENT...,CENTRO,49.0
1,01/01/2024 0:00,1:59:17,20240101-0002,07NC10,SUPERFICIE,M30,NC,Via de Servicio,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,seco,FLUIDA,0,0,0,"AVISO DE ALCANCE ENTRE 2 VEHÍCULOS, ANTES DE L...",USERA,NaN
2,01/01/2024 0:00,5:58:11,20240101-0007,03FT45,TÚNEL,A5,FT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,FLUIDA,0,0,0,ACCIDENTE LEVE ENTRE DOS VEHICULOS (ALCANCE) E...,VICALVARO,29.0
3,01/02/2024,9:51:53,20240102-0009,30RJ10,SUPERFICIE,M30,30RJ,RAMAL,Salida de via,COLISIÓN MEDIANA,...,0,0,Seco y limpio,FLUIDA,0,0,0,"ACCIDENTE LEVE SIN HERIDOS,EL VEHÍCULO CHOCA C...",RETIRO,NaN
4,02/01/2024 0:00,18:14:56,20240102-0022,11NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,DENSA,0,0,0,SEÑALIZAR COLISIÓN POR ALCANCE ENTRE DOS VEHÍC...,ARGANZUELA,50.0


## Helpers

Shared normalisation + junk handling used by every categorical clean.


In [2]:
JUNK = {"en blanco", "enblanco", "0", "", "nan", "none", "null", "na"}


def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_key(x) -> str | None:
    """Lowercase, strip accents, punctuation, and extra whitespace. None for junk/missing."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    key = strip_accents(s).lower()
    key = re.sub(r"[^a-z0-9 ]", "", key)
    key = re.sub(r"\s+", " ", key).strip()
    if key in JUNK:
        return None
    return key


def apply_map(series: pd.Series, mapping: dict) -> pd.Series:
    """Junk → NaN. Mapped keys use the dict. Other keys collapse to the majority spelling."""
    keys = series.map(norm_key)
    stripped = series.map(lambda v: str(v).strip())

    majority = {}
    for key, grp in stripped.groupby(keys, dropna=True):
        counts = grp.value_counts()
        top = counts[counts == counts.max()].index
        majority[key] = sorted(top)[0]

    def label_for(key):
        if key is None or (isinstance(key, float) and pd.isna(key)):
            return np.nan
        if key in mapping:
            return mapping[key]
        return majority[key]

    return keys.map(label_for).astype("object")


def before_after(df_before, df_after, cols):
    rows = []
    for c in cols:
        rows.append({
            "column": c,
            "nunique_before": int(df_before[c].nunique(dropna=False)),
            "nunique_after": int(df_after[c].nunique(dropna=False)),
            "missing_before": int(df_before[c].isna().sum()),
            "missing_after": int(df_after[c].isna().sum()),
        })
    return pd.DataFrame(rows)


CLEAN_COLS = [
    "estado_firme", "tipo_via", "tipo_colision", "tipo_accidente",
    "ubicacion", "circulacion", "carretera", "distrito",
]
snapshot_before = {c: df[c].copy() for c in CLEAN_COLS if c in df.columns}
print("helpers ready; columns to clean:", list(snapshot_before))


helpers ready; columns to clean: ['estado_firme', 'tipo_via', 'tipo_colision', 'tipo_accidente', 'ubicacion', 'circulacion', 'carretera', 'distrito']


## 1. Inspect unique values (before cleaning)

Confirm dirty variants before applying explicit maps.


In [3]:
for c in CLEAN_COLS:
    if c not in df.columns:
        continue
    vc = df[c].astype("object").value_counts(dropna=False)
    print(f"\n=== {c} ({vc.shape[0]} levels) ===")
    for val, n in vc.items():
        print(f"  {n:4d} | {val!r}")



=== estado_firme (13 levels) ===
  1618 | 'Seco y limpio'
   196 | 'Mojado'
    50 | nan
    40 | 'Seco'
    37 | 'SECO'
    32 | 'seco'
    32 | 'Otros'
    30 | 'SECO Y LIMPIO'
     9 | 'Húmedo'
     8 | 'MOJADO'
     6 | 'mojado'
     2 | '(En blanco)'
     2 | 'HUMEDO'

=== tipo_via (12 levels) ===
   825 | 'TRONCO'
   685 | 'VIA DE SERVICIO'
   416 | 'RAMAL'
    20 | 'Ramal'
    19 | 'TRANSFER'
    17 | 'TABLERO'
    16 | 'Via de Servicio'
    15 | 'Transfer'
    14 | 'tronco'
    12 | 'Túnel'
    12 | 'TUNEL'
    11 | 'Tronco'

=== tipo_colision (6 levels) ===
  1149 | 'COLISIÓN POR ALCANCE'
   393 | 'Otro'
   315 | 'COLISIÓN LATERAL'
    92 | 'COLISIÓN MEDIANA'
    80 | nan
    33 | 'COLISIÓN FRONTAL'

=== tipo_accidente (6 levels) ===
  1498 | 'Colision entre vehiculos'
   361 | 'Salida de via'
   173 | 'Otros'
    14 | 'Vuelco'
    13 | 'Colisión obstáculo'
     3 | 'Atropello'

=== ubicacion (2 levels) ===
  1733 | 'SUPERFICIE'
   329 | 'TÚNEL'

=== circulacion (6 levels) ==

## Remove duplicate IDs

100 IDs appear twice (same accident, not two crashes). Keep one row per `numero_incidencia`: the copy with more filled fields, then the earlier row. Prefer a real count over `-1`. Print field conflicts, then assert IDs are unique.

In [4]:
dups = df[df["numero_incidencia"].duplicated(keep=False)].sort_values("numero_incidencia")
print(len(dups), "rows,", dups["numero_incidencia"].nunique(), "IDs")
dups

200 rows, 100 IDs


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,motos_implicadas,pesados_implicados,estado_firme,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id
26,06/01/2024 0:00,18:47:21,20240106-0015,08NL90,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,DENSA,0,0,0,COLISIÓN POR ALCANCE DE TRES VEHÍCULOS EN CARR...,CENTRO,9.0
1975,06/01/2024 0:00,18:47:21,20240106-0015,08NL90,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,DENSA,0,0,0,COLISIÓN POR ALCANCE DE TRES VEHÍCULOS EN CARR...,CENTRO,9.0
52,10/01/2024 0:00,18:40:47,20240110-0037,03NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Mojado,CONGESTIONADA,0,0,0,Accidente sin heridos en arcén derecho. Vehícu...,CIUDAD LINEAL,36.0
1990,10/01/2024 0:00,18:40:47,20240110-0037,03NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Mojado,CONGESTIONADA,0,0,0,Accidente sin heridos en arcén derecho. Vehícu...,CIUDAD LINEAL,36.0
75,15/01/2024 0:00,8:41:53,20240115-0009,11NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Mojado,CONGESTIONADA,0,0,0,ACCIDENTE LEVE. A MI LLEGADA ME ENCUENTRO UN A...,USERA,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1913,20/12/2024 0:00,16:18:57,20241220-0038,00XC00,SUPERFICIE,M30,XC,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,0,0,Seco y limpio,FLUIDA,0,0,0,"ACCIDENTE EN CARRIL DERECHO, A MI LLEGADA NO ...",GETAFE,NaN
2000,26/12/2024 0:00,16:40:48,20241226-0028,14XC38,TÚNEL,M30,XC,TRONCO,Salida de via,COLISIÓN FRONTAL,...,0,0,Seco y limpio,DENSA,0,0,0,RECOGEMOS OBJETO EN CALZADA 4 HITOS QUE PRODUC...,TETUAN,1.0
1947,26/12/2024 0:00,16:40:48,20241226-0028,14XC38,TÚNEL,M30,XC,TRONCO,Salida de via,COLISIÓN FRONTAL,...,0,0,Seco y limpio,DENSA,0,0,0,RECOGEMOS OBJETO EN CALZADA 4 HITOS QUE PRODUC...,TETUAN,1.0
1979,27/12/2024 0:00,20:41:12,20241227-0039,05UL10,SUPERFICIE,A4,UL,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,1,0,Seco y limpio,DENSA,1,0,0,SEÑALIZO ALCANCE ENTRE TURISMO Y MOTOCICLETA. ...,FUENCARRAL,9.0


In [5]:
def is_missing(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    return s == "" or s == "-1"


def n_filled(row):
    return sum(not is_missing(v) for v in row)


n_before = len(df)
keep_idx = []
conflicts = []
filled_from_dup = []

for nid, g in df.groupby("numero_incidencia", sort=False):
    if len(g) == 1:
        keep_idx.append(g.index[0])
        continue
    filled = g.apply(n_filled, axis=1)
    best = filled[filled == filled.max()].index.min()
    keep_idx.append(best)
    for i, row in g.iterrows():
        if i == best:
            continue
        for col in g.columns:
            a, b = df.loc[best, col], row[col]
            if is_missing(a) and is_missing(b):
                continue
            if is_missing(a) and not is_missing(b):
                df.loc[best, col] = b
                filled_from_dup.append((nid, col, b))
                continue
            if not is_missing(b) and str(a).strip() != str(b).strip():
                conflicts.append((nid, col, a, b))

df = df.loc[sorted(keep_idx)].copy()
assert df["numero_incidencia"].is_unique, "duplicate IDs remain"

print(f"rows {n_before} → {len(df)}  (dropped {n_before - len(df)})")
print(f"cells filled from the dropped row: {len(filled_from_dup)}")
if filled_from_dup:
    print(pd.DataFrame(filled_from_dup, columns=["numero_incidencia", "column", "value"]).to_string(index=False))
print(f"conflicting fields logged: {len(conflicts)}")
if conflicts:
    conf = pd.DataFrame(conflicts, columns=["numero_incidencia", "column", "kept", "dropped"])
    print(conf.to_string(index=False))

rows 2062 → 1962  (dropped 100)
cells filled from the dropped row: 3
numero_incidencia        column                value
    20240510-0027  estado_firme        Seco y limpio
    20240615-0007  estado_firme        Seco y limpio
    20241003-0020 tipo_colision COLISIÓN POR ALCANCE
conflicting fields logged: 45
numero_incidencia       column            kept         dropped
    20240115-0009 estado_firme          Mojado          MOJADO
    20240203-0004     distrito        ALCORCON           USERA
    20240205-0004        fecha      02/05/2024 05/02/2024 0:00
    20240302-0019 estado_firme          Mojado          Húmedo
    20240313-0022     tipo_via           RAMAL           Ramal
    20240417-0017 estado_firme            SECO   Seco y limpio
    20240423-0020        fecha 23/04/2024 0:00      04/23/2024
    20240429-0037 estado_firme            SECO            Seco
    20240429-0037     distrito        MOSTOLES          RETIRO
    20240507-0044 estado_firme   SECO Y LIMPIO   Seco y lim

## 2. Normalise categoricals

Every categorical column is grouped by `norm_key` (lowercase, accents and punctuation removed). Junk becomes NaN. Keys listed in a map use that label. Any other key keeps the most common spelling already in the column, so case, accent, and punctuation variants of the same value collapse without a dictionary entry.

`carretera`: drop hyphens, uppercase. `M-607` → `M607`


In [6]:
ESTADO_FIRME_MAP = {
    "seco y limpio": "Seco y limpio",
    "seco": "Seco y limpio",
    "mojado": "Mojado",
    "humedo": "Húmedo",
}

TIPO_VIA_MAP = {
    "tronco": "TRONCO",
    "via de servicio": "VIA DE SERVICIO",
    "ramal": "RAMAL",
    "transfer": "TRANSFER",
    "tunel": "TUNEL",
}

MAPS = {
    "estado_firme": ESTADO_FIRME_MAP,
    "tipo_via": TIPO_VIA_MAP,
}

for col in CLEAN_COLS:
    if col not in df.columns:
        continue
    if col == "carretera":
        def clean_carretera(v):
            if norm_key(v) is None:
                return np.nan
            s = str(v).strip().upper().replace("-", "")
            return s

        df[col] = df[col].map(clean_carretera)
    else:
        df[col] = apply_map(df[col], MAPS.get(col, {}))

print("After categorical clean:")
for c in CLEAN_COLS:
    if c not in df.columns:
        continue
    print(f"\n=== {c} ===")
    print(df[c].value_counts(dropna=False).to_string())


After categorical clean:

=== estado_firme ===
estado_firme
Seco y limpio    1670
Mojado            203
NaN                48
Otros              32
Húmedo              9

=== tipo_via ===
tipo_via
TRONCO             803
VIA DE SERVICIO    669
RAMAL              419
TRANSFER            31
TUNEL               23
TABLERO             17

=== tipo_colision ===
tipo_colision
COLISIÓN POR ALCANCE    1092
Otro                     383
COLISIÓN LATERAL         304
COLISIÓN MEDIANA          83
NaN                       71
COLISIÓN FRONTAL          29

=== tipo_accidente ===
tipo_accidente
Colision entre vehiculos    1424
Salida de via                343
Otros                        166
Vuelco                        14
Colisión obstáculo            12
Atropello                      3

=== ubicacion ===
ubicacion
SUPERFICIE    1651
TÚNEL          311

=== circulacion ===
circulacion
FLUIDA           689
CONGESTIONADA    615
DENSA            573
NaN               85

=== carretera ===
carretera
M30 

## 3. Numeric fixes, dates, and hora_decimal

In [7]:
COUNT_COLS = [
    "vehiculos_implicados", "motos_implicadas", "pesados_implicados",
    "heridos_leves", "heridos_graves", "victimas_mortales",
]

minus_one_log = {}
for c in COUNT_COLS:
    if c not in df.columns:
        continue
    df[c] = pd.to_numeric(df[c], errors="coerce")
    n_m1 = int((df[c] == -1).sum())
    minus_one_log[c] = n_m1
    if n_m1:
        df.loc[df[c] == -1, c] = np.nan

print("-1 → NaN counts:", minus_one_log)
print("vehiculos_implicados == 0:", int((df["vehiculos_implicados"] == 0).sum()))


def parse_fecha(val):
    """Parse mixed DMY / YMD / MDY date strings (date part only)."""
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip().split()[0]
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT


df["fecha_calendario"] = df["fecha"].map(parse_fecha)
print(
    f"fecha_calendario parsed: {df['fecha_calendario'].notna().sum()} / {len(df)} "
    f"(failed {int(df['fecha_calendario'].isna().sum())})"
)
print("fecha range:", df["fecha_calendario"].min(), "→", df["fecha_calendario"].max())

hora = pd.to_datetime(df["hora"].astype(str), format="%H:%M:%S", errors="coerce")
if hora.isna().mean() > 0.5:
    hora = pd.to_datetime(df["hora"].astype(str), format="%H:%M", errors="coerce")
df["hora_decimal"] = (hora.dt.hour + hora.dt.minute / 60).astype("Float64")
print(f"hora_decimal parsed: {df['hora_decimal'].notna().sum()} / {len(df)}")


-1 → NaN counts: {'vehiculos_implicados': 14, 'motos_implicadas': 0, 'pesados_implicados': 0, 'heridos_leves': 13, 'heridos_graves': 0, 'victimas_mortales': 0}
vehiculos_implicados == 0: 17
fecha_calendario parsed: 1962 / 1962 (failed 0)
fecha range: 2024-01-01 00:00:00 → 2024-12-31 00:00:00
hora_decimal parsed: 1962 / 1962


## 4. Label: `grave`

`grave = 1` if `heridos_graves > 0` **or** `victimas_mortales > 0`.


In [8]:
df["grave"] = (
    (df["heridos_graves"].fillna(0) > 0) | (df["victimas_mortales"].fillna(0) > 0)
).astype(int)

n = len(df)
n_sev = int(df["grave"].sum())
print(f"grave: {n_sev} / {n} ({100 * n_sev / n:.2f}%)")
print("heridos_graves > 0:", int((df["heridos_graves"].fillna(0) > 0).sum()))
print("victimas_mortales > 0:", int((df["victimas_mortales"].fillna(0) > 0).sum()))
print("both:", int(((df["heridos_graves"].fillna(0) > 0) & (df["victimas_mortales"].fillna(0) > 0)).sum()))
df["grave"].value_counts()


grave: 45 / 1962 (2.29%)
heridos_graves > 0: 40
victimas_mortales > 0: 7
both: 2


grave
0    1917
1      45
Name: count, dtype: int64

## 5. Data report

Before/after unique counts, missingness after cleaning, label balance, and decision list.


In [9]:
df_before_cats = pd.DataFrame(snapshot_before)
report_ba = before_after(df_before_cats, df, list(snapshot_before))
print("=== Before / after unique & missing (cleaned categoricals) ===")
display(report_ba)

miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(2),
    "nunique": df.nunique(dropna=False),
    "dtype": df.dtypes.astype(str),
}).sort_values("pct_missing", ascending=False)
print("\n=== Missing % after cleaning (all columns) ===")
display(miss)

print(f"\nRows: {len(df)}")
print(f"Label balance: grave={n_sev} ({100 * n_sev / n:.2f}%), no grave={n - n_sev} ({100 * (n - n_sev) / n:.2f}%)")


=== Before / after unique & missing (cleaned categoricals) ===


,column,nunique_before,nunique_after,missing_before,missing_after
0,estado_firme,13,5,50,48
1,tipo_via,12,6,0,0
2,tipo_colision,6,6,80,71
3,tipo_accidente,6,6,0,0
4,ubicacion,2,2,0,0
5,circulacion,6,4,75,85
6,carretera,14,13,1,1
7,distrito,25,25,0,0



=== Missing % after cleaning (all columns) ===


,n_missing,pct_missing,nunique,dtype
unidad_samur_id,533,27.17,51,float64
descripcion,102,5.20,1801,object
circulacion,85,4.33,4,object
tipo_colision,71,3.62,6,object
estado_firme,48,2.45,5,object
vehiculos_implicados,14,0.71,9,float64
heridos_leves,13,0.66,7,float64
carretera,1,0.05,13,object
fecha,0,0.00,440,object
hora_decimal,0,0.00,921,Float64



Rows: 1962
Label balance: grave=45 (2.29%), no grave=1917 (97.71%)


## 6. Save cleaned table

Writes (`./data/accidentes_clean.csv`).


In [10]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("wrote", OUT_PATH.resolve())
print("shape", df.shape)
print("columns:", list(df.columns))
df.head(10)


wrote /Users/lukas/dev/UC3M/Project-UC3M/1_2_severity/data/accidentes_clean.csv
shape (1962, 24)
columns: ['fecha', 'hora', 'numero_incidencia', 'localizacion', 'ubicacion', 'carretera', 'eje', 'tipo_via', 'tipo_accidente', 'tipo_colision', 'vehiculos_implicados', 'motos_implicadas', 'pesados_implicados', 'estado_firme', 'circulacion', 'heridos_leves', 'heridos_graves', 'victimas_mortales', 'descripcion', 'distrito', 'unidad_samur_id', 'fecha_calendario', 'hora_decimal', 'grave']


,fecha,hora,numero_incidencia,localizacion,ubicacion,carretera,eje,tipo_via,tipo_accidente,tipo_colision,...,circulacion,heridos_leves,heridos_graves,victimas_mortales,descripcion,distrito,unidad_samur_id,fecha_calendario,hora_decimal,grave
0,01/01/2024 0:00,1:30:48,20240101-0001,08NL10,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,NaN,1.0,0,0,SEÑALIZAR ACCIDENTE POR ALCANCE EN CARRIL CENT...,CENTRO,49.0,2024-01-01,1.5,0
1,01/01/2024 0:00,1:59:17,20240101-0002,07NC10,SUPERFICIE,M30,NC,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,"AVISO DE ALCANCE ENTRE 2 VEHÍCULOS, ANTES DE L...",USERA,NaN,2024-01-01,1.983333,0
2,01/01/2024 0:00,5:58:11,20240101-0007,03FT45,TÚNEL,A5,FT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,ACCIDENTE LEVE ENTRE DOS VEHICULOS (ALCANCE) E...,VICALVARO,29.0,2024-01-01,5.966667,0
3,01/02/2024,9:51:53,20240102-0009,30RJ10,SUPERFICIE,M30,30RJ,RAMAL,Salida de via,COLISIÓN MEDIANA,...,FLUIDA,0.0,0,0,"ACCIDENTE LEVE SIN HERIDOS,EL VEHÍCULO CHOCA C...",RETIRO,NaN,2024-02-01,9.85,0
4,02/01/2024 0:00,18:14:56,20240102-0022,11NL00,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,DENSA,0.0,0,0,SEÑALIZAR COLISIÓN POR ALCANCE ENTRE DOS VEHÍC...,ARGANZUELA,50.0,2024-01-02,18.233333,0
5,02/01/2024 0:00,19:34:29,20240102-0026,04AT90,SUPERFICIE,A2,AT,TRONCO,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,CONGESTIONADA,0.0,0,0,AVISO DE CENTRO DE CONTROL PARA SEÑALIZAR ACCI...,SAN BLAS,39.0,2024-01-02,19.566667,0
6,02/01/2024 0:00,20:14:50,20240102-0028,09NL50,SUPERFICIE,M30,NL,VIA DE SERVICIO,Colisión obstáculo,Otro,...,CONGESTIONADA,0.0,0,0,SEÑALIZAR ACCIDENTE POR PERDIDA DE CARGA DE FU...,CENTRO,NaN,2024-01-02,20.233333,0
7,02/01/2024 0:00,20:24:15,20240102-0029,08RT20,SUPERFICIE,M30,08RT,RAMAL,Colision entre vehiculos,COLISIÓN POR ALCANCE,...,FLUIDA,0.0,0,0,AVISO DE CENTRO DE CONTROL PARA SEÑALIZAR CAID...,HORTALEZA,36.0,2024-01-02,20.4,0
8,02/01/2024 0:00,21:40:06,20240102-0030,08RT11,SUPERFICIE,M30,08RT,RAMAL,Salida de via,Otro,...,FLUIDA,0.0,0,0,VEHÍCULO SE SALE DE VIA COLISIONANDO CONTRA BA...,VALLECAS,NaN,2024-01-02,21.666667,0
9,02/01/2024 0:00,22:37:53,20240102-0031,07XL40,SUPERFICIE,M30,XL,VIA DE SERVICIO,Colision entre vehiculos,COLISIÓN LATERAL,...,FLUIDA,0.0,0,0,SEÑALIZACION DE ACCIDENTE POR ALCANCE LATERAL ...,CARABANCHEL,5.0,2024-01-02,22.616667,0
